# 02b - PCA, NMF and UMAP

Three complementary views of the same pixels: PCA for variance structure,
NMF for additive chemical endmembers, UMAP for non-linear neighbourhood
structure.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [ ]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
print("Raw data :", config.raw_dir)
print("Outputs  :", config.processed_dir)

In [ ]:
from src.features import decomposition as decomp
from src.features.selection import build_cube, select_analysis_channels
from src.preprocessing.stack_io import load_clean_stack

images, metadata = load_clean_stack(config.processed_dir)
selection = select_analysis_channels(images, metadata, config)
cube = build_cube(images, selection)

prepared = decomp.prepare_pixels(cube)
print(f"{prepared.n_foreground:,} foreground pixels of {len(prepared.fg_mask):,}")

## PCA

Fitted on standardized intensities so no single high-count channel dominates.

In [ ]:
from src.viz import decomposition_plots

pca = decomp.run_pca(prepared, random_state=42)
print("explained variance (%):", (pca["explained_variance"] * 100).round(2))

decomposition_plots.plot_pca_scree(
    pca, selection.labels, config.figure_path("02b_pca_scree.png")
)
decomposition_plots.plot_pca_component_maps(
    pca, prepared.fg_mask, config.figure_path("02b_pca_component_maps.png")
)

## NMF

Non-negativity makes the components physically interpretable: each pixel is a
sum of endmember contributions, never a cancellation.

The rank is chosen from the elbow of the reconstruction-error curve rather
than fixed, so a new dataset gets its own appropriate rank.

In [ ]:
sweep = decomp.run_nmf_sweep(prepared, sweep=config.get("decomposition.nmf.sweep"))
for k in sorted(sweep):
    print(f"  k={k}: reconstruction error {sweep[k]['error']:,.1f}")

chosen_k = decomp.select_nmf_rank(sweep)
print(f"\nelbow -> k = {chosen_k}")

In [ ]:
nmf = decomp.run_nmf(prepared, n_components=chosen_k, sweep_results=sweep)

decomposition_plots.plot_nmf_model_selection(
    sweep, chosen_k, config.figure_path("02b_nmf_model_selection.png")
)
decomposition_plots.plot_nmf_endmembers(
    nmf, selection.labels, config.figure_path("02b_nmf_endmembers.png")
)
decomposition_plots.plot_nmf_abundance_maps(
    nmf, config.figure_path("02b_nmf_abundance_maps.png")
)
decomposition_plots.plot_nmf_rgb_composite(
    nmf, config.figure_path("02b_nmf_rgb_composite.png")
)

In [ ]:
import pandas as pd

# Endmember spectra: which masses define each component.
pd.DataFrame(
    nmf["H"],
    index=[f"EM{i + 1}" for i in range(chosen_k)],
    columns=selection.labels,
).round(2)

## UMAP

Only foreground pixels are embedded; background is a single degenerate point
that would otherwise dominate the neighbourhood graph.

In [ ]:
umap_result = None
if config.get("decomposition.umap.enabled", True):
    umap_result = decomp.run_umap(prepared)
    decomposition_plots.plot_umap_scatter(
        umap_result, prepared, nmf, config.figure_path("02b_umap_scatter.png")
    )
    decomposition_plots.plot_umap_spatial_maps(
        umap_result, config.figure_path("02b_umap_spatial_maps.png")
    )
    print(f"embedded {umap_result['embedding'].shape[0]:,} pixels")

## Save for the next stage

In [ ]:
written = decomp.save_decomposition(
    config.processed_dir, prepared, pca, nmf, umap_result,
    selection.labels, selection.keys,
)
for name, path in written.items():
    print(f"{name:18s} -> {path.name}")

Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage decompose
```